# OCR Model Selection Lab — Colab CPU/GPU

Ce notebook sert à lancer le benchmark dans une machine Colab propre.

Point important : le repo GitHub n’est pas le modèle. Le repo contient le banc de test : interface, métriques, runner, graphiques, lecture du dataset et export des résultats. Colab démarre vide ; il doit donc récupérer ce code depuis GitHub ou depuis un ZIP.

Workflow simple :

1. Colab récupère le code du benchmark depuis GitHub.
2. Colab installe les dépendances Python.
3. Colab charge le dataset inclus ou un dataset importé.
4. Colab prépare les modèles à tester : EasyOCR, adaptateur Python/API, ou ressource Kaggle.
5. Colab lance le benchmark et produit les métriques.

Le GPU accélère seulement les adaptateurs compatibles. EasyOCR utilise CUDA automatiquement si Colab fournit un GPU. Les tokens/s concernent uniquement les modèles génératifs qui exposent leurs compteurs.

In [ ]:
# ÉTAPE 1 — Dire à Colab où récupérer le code du benchmark.
# Recommandé : cloner le repo GitHub privé avec un Personal Access Token en lecture.
# Plan B : mettre USE_PRIVATE_GITHUB_REPO=False pour uploader un ZIP du projet.
REPO_URL = "https://github.com/Eric-Kambire/ocr-model-selection-lab.git"
USE_PRIVATE_GITHUB_REPO = True
BRANCH = "main"
PROJECT_DIR = "/content/ocr-model-selection-lab"


In [ ]:
# ÉTAPE 2 — Télécharger le code du benchmark dans la machine Colab.
# Si le repo est privé, Colab demande un token GitHub. Le token n’est pas écrit dans le notebook.
import base64, getpass, os, pathlib, shutil, subprocess, zipfile

project = pathlib.Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)

if USE_PRIVATE_GITHUB_REPO:
    token = getpass.getpass("GitHub PAT (lecture du dépôt privé) : ")
    credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}"})
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True, env=git_env)
    finally:
        del token, credential, git_env
else:
    from google.colab import files
    print("Chargez le ZIP du dépôt OCR Model Selection Lab.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if not zip_name:
        raise ValueError("Aucun fichier ZIP reçu.")
    project.mkdir(parents=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(project)
    candidates = [p.parent for p in project.rglob("main.py")]
    if len(candidates) != 1:
        raise RuntimeError(f"Impossible d’identifier la racine : {candidates}")
    project = candidates[0]

os.chdir(project)
print("Projet chargé depuis :", project)

In [ ]:
# ÉTAPE 3 — Installer les dépendances du benchmark et KaggleHub.
# KaggleHub permet de télécharger des datasets/modèles Kaggle depuis Colab si besoin.
!python -m pip install -q -r requirements-ocr.txt kagglehub


In [ ]:
# ÉTAPE 4 — Vérifier si Colab tourne en CPU ou en GPU.
import platform, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python :", platform.python_version())
print("PyTorch :", torch.__version__)
print("Matériel :", DEVICE)
if DEVICE == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA :", torch.version.cuda)
else:
    print("CPU actif. Pour tester le GPU : Exécution > Modifier le type d’exécution > T4 GPU.")

## Gérer le dataset dans Colab

Par défaut, le notebook utilise le dataset inclus dans le dépôt : `dataset/dataset.json` + les images dans `dataset/`.

Pour tester vos propres documents dans Colab, préparez un ZIP avec ce format :

```text
mon_dataset.zip
├── labels.csv
└── images/
    ├── cheque_001.png
    └── form_001.jpg
```

`labels.csv` doit contenir au minimum :

```csv
image_path,ground_truth,category,description
images/cheque_001.png,"texte attendu exact",bank,"chèque manuscrit"
images/form_001.jpg,"texte attendu exact",handwritten_form,"formulaire rempli"
```

Alternative : remplacez `labels.csv` par un `dataset.json` contenant une liste d’objets avec les mêmes champs. Les images importées sont copiées dans `dataset/user_uploads/`, puis ajoutées au catalogue Colab.

In [ ]:
# ÉTAPE 5 — Dataset.
# Par défaut, on utilise le dataset inclus dans le repo.
# Mettez True pour uploader un dataset ZIP depuis votre ordinateur dans la session Colab.
IMPORT_CUSTOM_DATASET_ZIP = False

import json, pathlib, shutil, zipfile
import pandas as pd

from main import ROOT_DIR, load_dataset
from ocr_benchmark.dataset_repository import DatasetRepository


def _read_custom_records(extract_dir: pathlib.Path):
    labels_csv = next(extract_dir.rglob("labels.csv"), None)
    dataset_json = next(extract_dir.rglob("dataset.json"), None)

    if labels_csv:
        table = pd.read_csv(labels_csv).fillna("")
        records = table.to_dict("records")
        base_dir = labels_csv.parent
    elif dataset_json:
        records = json.loads(dataset_json.read_text(encoding="utf-8"))
        base_dir = dataset_json.parent
    else:
        raise ValueError("Le ZIP doit contenir labels.csv ou dataset.json.")

    required = {"image_path", "ground_truth", "category"}
    for index, record in enumerate(records, start=1):
        missing = required - set(record)
        if missing:
            raise ValueError(f"Ligne {index}: champs manquants {sorted(missing)}")
    return records, base_dir


def _assert_inside(child: pathlib.Path, parent: pathlib.Path) -> pathlib.Path:
    resolved_child = child.resolve()
    resolved_parent = parent.resolve()
    if resolved_child != resolved_parent and resolved_parent not in resolved_child.parents:
        raise ValueError(f"Chemin non autorisé hors du dataset : {child}")
    return resolved_child


def _safe_extract_zip(zip_name: str, extract_dir: pathlib.Path) -> None:
    with zipfile.ZipFile(zip_name) as archive:
        for member in archive.infolist():
            target = _assert_inside(extract_dir / member.filename, extract_dir)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)


def _resolve_image(base_dir: pathlib.Path, image_path: str) -> pathlib.Path:
    candidate = _assert_inside(base_dir / image_path, base_dir)
    if candidate.is_file():
        return candidate

    matches = list(base_dir.rglob(pathlib.Path(image_path).name))
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"Image introuvable dans le ZIP : {image_path}")


def import_custom_dataset_zip():
    from google.colab import files

    print("Chargez votre ZIP dataset contenant labels.csv ou dataset.json.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if not zip_name:
        raise ValueError("Aucun ZIP reçu.")

    extract_dir = pathlib.Path("/content/custom_dataset_upload")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)

    _safe_extract_zip(zip_name, extract_dir)

    records, base_dir = _read_custom_records(extract_dir)
    repository = DatasetRepository(ROOT_DIR)
    added = []

    for record in records:
        image_file = _resolve_image(base_dir, str(record["image_path"]))
        added.append(
            repository.add_labeled_image(
                image_file,
                str(record["ground_truth"]),
                str(record["category"]),
                str(record.get("description", "Dataset importé depuis Colab")),
            )
        )

    print(f"{len(added)} document(s) ajoutés au dataset Colab.")
    return added


if IMPORT_CUSTOM_DATASET_ZIP:
    import_custom_dataset_zip()

dataset_preview = load_dataset()
print(f"Dataset actif : {len(dataset_preview)} document(s)")
display(pd.Series([item["category"] for item in dataset_preview]).value_counts().rename("documents").to_frame())

## Installer et télécharger les modèles

Trois familles sont prévues :

- `mock:*` : modèle simulé, rapide, sert à vérifier que le benchmark fonctionne.
- `easyocr:EasyOCR-Local` : moteur OCR classique recommandé pour Colab. Il utilise le GPU automatiquement si disponible.
- futurs adaptateurs Python/API : meilleur chemin pour brancher des modèles Hugging Face, cloud ou propriétaires.
- `ollama:<nom-du-modèle>` : option avancée seulement si vous voulez lancer un serveur Ollama dans Colab. Ce n’est pas nécessaire pour utiliser Colab.

Conseil pratique : commencez avec `mock`, puis `easyocr`. N’activez Ollama que si vous savez explicitement que le modèle à tester doit tourner via Ollama.

In [ ]:
# ÉTAPE 6 — Préparer les modèles locaux simples.
# Choisissez ici les moteurs à préparer dans Colab.
# Chemin recommandé : EasyOCR ou adaptateurs Python/API.
# EasyOCR télécharge automatiquement ses poids au premier lancement.
# Ollama est optionnel : gardez False sauf besoin explicite d’un serveur Ollama dans Colab.
INSTALL_EASYOCR = True
INSTALL_OLLAMA = False

# Option avancée Ollama seulement.
# Exemples de modèles vision Ollama :
# - "llama3.2-vision" : bon modèle vision, plus lourd.
# - "moondream" : plus léger, utile pour un premier test.
# Changez cette liste selon les modèles que vous voulez benchmarker.
OLLAMA_MODELS_TO_PULL = ["moondream"]

import os, subprocess, textwrap, time

if INSTALL_EASYOCR:
    print("EasyOCR est disponible via requirements-ocr.txt. Les poids seront téléchargés au premier benchmark.")

if INSTALL_OLLAMA:
    print("Installation d’Ollama dans Colab…")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

    print("Démarrage du serveur Ollama en arrière-plan…")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
    )
    time.sleep(8)

    for model_name in OLLAMA_MODELS_TO_PULL:
        print(f"Téléchargement du modèle Ollama : {model_name}")
        subprocess.run(["ollama", "pull", model_name], check=True)

    print("Modèles Ollama installés :")
    subprocess.run(["ollama", "list"], check=True)
else:
    print("Ollama ignoré. C’est normal sur Colab sauf si vous voulez explicitement tester un modèle via Ollama.")

## Importer une ressource Kaggle si nécessaire

Kaggle peut fournir deux choses différentes :

- un dataset : images + labels à ajouter au benchmark ;
- un modèle : fichiers de poids/configuration à charger par un adaptateur Python.

Télécharger un modèle Kaggle ne suffit pas à le comparer automatiquement. Le benchmark a besoin d’un adaptateur qui sait appeler ce modèle et retourner au minimum : texte extrait, temps, statut, sortie brute. C’est volontaire : ça garde la comparaison propre et traçable.

Si vous avez un handle Kaggle, collez-le ci-dessous. Exemple de handle : `owner/dataset-name` ou `owner/model-name/framework/variation/version` selon le type de ressource.

In [ ]:
# ÉTAPE 7 — Optionnel : télécharger un dataset ou modèle depuis Kaggle.
# Laissez les deux variables à False si vous utilisez seulement le dataset inclus et EasyOCR.
DOWNLOAD_KAGGLE_DATASET = False
DOWNLOAD_KAGGLE_MODEL = False

# À remplir seulement si vous activez une option Kaggle.
KAGGLE_DATASET_HANDLE = ""  # exemple : "senju14/ocr-dataset-of-multi-type-documents"
KAGGLE_MODEL_HANDLE = ""    # exemple : "owner/model/framework/variation/version"

import kagglehub

kaggle_dataset_path = None
kaggle_model_path = None

if DOWNLOAD_KAGGLE_DATASET:
    if not KAGGLE_DATASET_HANDLE:
        raise ValueError("Renseignez KAGGLE_DATASET_HANDLE avant de télécharger un dataset Kaggle.")
    kaggle_dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)
    print("Dataset Kaggle téléchargé ici :", kaggle_dataset_path)
    print("Ensuite, convertissez-le au format labels.csv ou dataset.json décrit plus haut.")

if DOWNLOAD_KAGGLE_MODEL:
    if not KAGGLE_MODEL_HANDLE:
        raise ValueError("Renseignez KAGGLE_MODEL_HANDLE avant de télécharger un modèle Kaggle.")
    kaggle_model_path = kagglehub.model_download(KAGGLE_MODEL_HANDLE)
    print("Modèle Kaggle téléchargé ici :", kaggle_model_path)
    print("Important : pour le benchmarker, il faut un adaptateur Python qui charge ce dossier.")

if not DOWNLOAD_KAGGLE_DATASET and not DOWNLOAD_KAGGLE_MODEL:
    print("Aucune ressource Kaggle téléchargée. C’est normal pour un premier test.")

## Configurer le benchmark

- `Tout le dataset` utilise tous les documents.
- `Quantité globale` prélève `GLOBAL_QUANTITY` documents.
- `Par catégorie` utilise les limites définies dans `CATEGORY_QUANTITIES`.
- `TIMEOUT_SECONDS` est la durée maximale officielle par image. Une sortie arrivée plus tard est conservée dans la trace, mais n’est pas notée.
- `MODEL_PROMPT` est envoyé aux modèles génératifs compatibles. EasyOCR et les modèles simulés l’ignorent.

In [ ]:
# ÉTAPE 8 — Choisir les modèles, le nombre de documents et le timeout.
# Modèles à tester. Gardez le préfixe fournisseur : mock:, easyocr:, ollama:, etc.
# Exemples :
# MODEL_SPECS = ["mock:MockOCR-Colab"]
# MODEL_SPECS = ["easyocr:EasyOCR-Local"]
# Option avancée si vous avez activé INSTALL_OLLAMA=True :
# MODEL_SPECS = ["ollama:moondream"]
MODEL_SPECS = ["mock:MockOCR-Colab"]

MODEL_PROMPT = """You are a professional layout-preserving OCR engine.
Transcribe all visible text in the image.
Return only the transcription, without explanation.
Preserve tables and line breaks when possible."""

SELECTION_MODE = "Quantité globale"  # Tout le dataset | Quantité globale | Par catégorie
GLOBAL_QUANTITY = 10
CATEGORY_QUANTITIES = {"bank": 3, "tables": 3, "handwritten": 4}
SHUFFLE = True
SEED = 42
TIMEOUT_SECONDS = 300
MAX_ERRORS = 0  # 0 = aucune limite
EVAL_MODE = "Standard"  # Standard | Bankmark


In [ ]:
# ÉTAPE 9 — Préparer la liste exacte des documents qui seront testés.
import pandas as pd
from main import RUNS_DIR, load_dataset, select_dataset_items
from ocr_benchmark.domain import BenchmarkCase
from ocr_benchmark.registry import build_default_registry
from ocr_benchmark.reporting import RunCheckpoint
from ocr_benchmark.runner import BenchmarkRunner, summarize_results

dataset = load_dataset()
counts = pd.Series([item["category"] for item in dataset]).value_counts()
category_rows = [
    [category, int(available), int(CATEGORY_QUANTITIES.get(category, 0))]
    for category, available in counts.items()
]
selected = select_dataset_items(
    dataset,
    SELECTION_MODE,
    GLOBAL_QUANTITY,
    category_rows,
    shuffle=SHUFFLE,
    seed=SEED,
)
print(f"{len(selected)} document(s), {len(MODEL_SPECS)} modèle(s), {len(selected) * len(MODEL_SPECS)} évaluation(s).")
display(pd.Series([item["category"] for item in selected]).value_counts().rename("documents").to_frame())

In [ ]:
# ÉTAPE 10 — Lancer le benchmark et sauvegarder les résultats.
runner = BenchmarkRunner(build_default_registry())
cases = [BenchmarkCase.from_dict(item) for item in selected]

def save_trace(event):
    RunCheckpoint(event["run_id"], RUNS_DIR).append_trace(event)

run_id, results = runner.run(
    MODEL_SPECS,
    cases,
    eval_mode=EVAL_MODE,
    timeout_seconds=TIMEOUT_SECONDS,
    max_errors=MAX_ERRORS,
    model_prompt=MODEL_PROMPT,
    trace=save_trace,
)
summary = summarize_results(results)
run_dir = RunCheckpoint(run_id, RUNS_DIR).finalize(summary, results)
print("Run ID :", run_id)
print("Résultats :", run_dir)
display(summary)

## Examiner les sorties et traces

`status=timeout` signifie que la durée maximale a été dépassée. Une ligne `timing=late_after_timeout` permet d’examiner la réponse reçue ensuite. Les champs bruts sont conservés pour audit avant nettoyage.

In [ ]:
# ÉTAPE 11 — Lire les traces brutes, y compris les réponses arrivées après timeout.
import json

trace_path = run_dir / "traces.jsonl"
traces = []
if trace_path.exists():
    traces = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line]
trace_table = pd.DataFrame(traces)
columns = [name for name in ["model", "image_path", "timing", "status", "latency", "text", "reasoning", "error"] if name in trace_table]
display(trace_table[columns] if columns else trace_table)


In [ ]:
# ÉTAPE 12 — Télécharger l’ensemble du run : CSV, JSON, rapport et traces brutes.
from google.colab import files
archive = shutil.make_archive(f"/content/{run_id}", "zip", run_dir)
files.download(archive)

## Interface complète

L’interface contient la sélection des quantités, la progression, l’image courante, les métriques en direct et les paramètres. Colab fournit une URL publique temporaire : ne l’utilisez pas avec des documents sensibles.

In [ ]:
# ÉTAPE 13 — Lancer l’interface Gradio complète dans Colab.
from main import build_ui

app = build_ui()
app.launch(share=True, debug=False)